# Dataset Analysis: Class Proportions Over Time

This notebook loads each dataset used in the quantification-over-time project, replicates the preprocessing from `project/data_loading.py` (while preserving timestamps), and visualises how class proportions evolve over time using interactive Plotly charts.

In [1]:
import pandas as pd
import numpy as np
import chardet
from pathlib import Path
import plotly.graph_objects as go
import plotly.io as pio

pio.renderers.default = "notebook_connected"

DATA_DIR = Path(".")

## Dataset Loading Functions

Each loader replicates the preprocessing from `project/data_loading.py` but **keeps the timestamp column** so we can plot class proportions over time.

All loaders return a DataFrame with at least two columns: `timestamp` and `label`.

In [2]:
def load_global_covid19_tweets() -> pd.DataFrame:
    """Merge train + test, map Sentiment to {-1, 0, 1}, parse TweetAt as datetime."""
    train_path = DATA_DIR / "global_covid19_tweets" / "global_covid19_tweets" / "Corona_NLP_train.csv"
    test_path  = DATA_DIR / "global_covid19_tweets" / "global_covid19_tweets" / "Corona_NLP_test.csv"

    df = pd.concat([pd.read_csv(test_path), pd.read_csv(train_path)])
    df = df[["Sentiment", "OriginalTweet", "TweetAt"]].copy()

    sentiment_map = {
        "Extremely Positive": 1, "Positive": 1,
        "Extremely Negative": -1, "Negative": -1,
        "Neutral": 0,
    }
    df["label"] = df["Sentiment"].map(sentiment_map)
    df = df.dropna(subset=["label"])
    df["label"] = df["label"].astype(int)

    df["timestamp"] = pd.to_datetime(df["TweetAt"], format="%d-%m-%Y")
    df = df.sort_values("timestamp").reset_index(drop=True)

    return df[["timestamp", "label"]]


def load_nepali_dataset_eng() -> pd.DataFrame:
    """Load Nepali dataset, keep Label {-1, 0, 1} and Datetime."""
    df = pd.read_csv(DATA_DIR / "Nepali_dataset_Eng.csv")
    df = df[df["Label"].isin([-1, 0, 1])].copy()
    df = df.rename(columns={"Label": "label"})

    df["timestamp"] = pd.to_datetime(df["Datetime"])
    df = df.sort_values("timestamp").reset_index(drop=True)

    return df[["timestamp", "label"]]


def load_apple_twitter_sentiment() -> pd.DataFrame:
    """Load Apple Twitter Sentiment, parse full Twitter date, map sentiment codes."""
    data_path = DATA_DIR / "Apple-Twitter-Sentiment-DFE.csv"
    with open(data_path, "rb") as f:
        enc = chardet.detect(f.read())

    df = pd.read_csv(data_path, encoding=enc["encoding"])
    df = df[df["sentiment"].astype(str).isin(["1", "3", "5"])].copy()

    sentiment_map = {"5": 1, "3": -1, "1": 0}
    df["label"] = df["sentiment"].astype(str).map(sentiment_map)

    df["timestamp"] = pd.to_datetime(df["date"], format="%a %b %d %H:%M:%S %z %Y", utc=True)
    df = df.sort_values("timestamp").reset_index(drop=True)

    return df[["timestamp", "label"]]


def load_bike() -> pd.DataFrame:
    """Load bike sharing hourly data, bin cnt into {0, 1}, keep dteday as timestamp."""
    df = pd.read_csv(DATA_DIR / "bike_sharing_dataset" / "hour.csv")

    bins = [0, 100, 1000]
    labels = [0, 1]
    df["label"] = pd.cut(df["cnt"], bins=bins, labels=labels).astype(int)

    df["timestamp"] = pd.to_datetime(df["dteday"])
    df = df.sort_values(["timestamp", "hr"]).reset_index(drop=True)

    return df[["timestamp", "label"]]

## Load All Datasets & Save Preprocessed CSVs

In [3]:
DATASET_LOADERS = {
    "global_covid19_tweets": load_global_covid19_tweets,
    "nepali_dataset_eng": load_nepali_dataset_eng,
    "Apple-Twitter-Sentiment-DFE": load_apple_twitter_sentiment,
    "bike": load_bike,
}

datasets: dict[str, pd.DataFrame] = {}

for name, loader in DATASET_LOADERS.items():
    df = loader()
    datasets[name] = df

    out_path = DATA_DIR / f"{name}_preprocessed.csv"
    df.to_csv(out_path, index=False)
    print(f"{name:>35s}  |  {len(df):>6,} rows  |  classes {sorted(df['label'].unique())}  |  saved -> {out_path.name}")

              global_covid19_tweets  |  44,954 rows  |  classes [-1, 0, 1]  |  saved -> global_covid19_tweets_preprocessed.csv
                 nepali_dataset_eng  |  33,435 rows  |  classes [-1, 0, 1]  |  saved -> nepali_dataset_eng_preprocessed.csv
        Apple-Twitter-Sentiment-DFE  |   3,804 rows  |  classes [-1, 0, 1]  |  saved -> Apple-Twitter-Sentiment-DFE_preprocessed.csv
                               bike  |  17,379 rows  |  classes [0, 1]  |  saved -> bike_preprocessed.csv


## Compute Class Proportions Over Time

For each dataset we group by the timestamp (date granularity), count each class, and compute proportions.

In [4]:
SENTIMENT_LABELS = {-1: "Negative", 0: "Neutral", 1: "Positive"}
BIKE_LABELS = {0: "Low (0\u2013100)", 1: "High (100\u20131000)"}

DATASET_CLASS_LABELS = {
    "global_covid19_tweets": SENTIMENT_LABELS,
    "nepali_dataset_eng": SENTIMENT_LABELS,
    "Apple-Twitter-Sentiment-DFE": SENTIMENT_LABELS,
    "bike": BIKE_LABELS,
}

proportions: dict[str, pd.DataFrame] = {}

for name, df in datasets.items():
    date_col = df["timestamp"].dt.date
    label_map = DATASET_CLASS_LABELS[name]

    counts = df.groupby([date_col, "label"]).size().unstack(fill_value=0)
    props = counts.div(counts.sum(axis=1), axis=0)
    props.index = pd.to_datetime(props.index)
    props = props.sort_index()

    props = props.rename(columns=label_map)

    proportions[name] = props
    print(f"{name}: {len(props)} unique dates, classes = {list(props.columns)}")

global_covid19_tweets: 44 unique dates, classes = ['Negative', 'Neutral', 'Positive']
nepali_dataset_eng: 335 unique dates, classes = ['Negative', 'Neutral', 'Positive']
Apple-Twitter-Sentiment-DFE: 10 unique dates, classes = ['Negative', 'Neutral', 'Positive']
bike: 731 unique dates, classes = ['Low (0–100)', 'High (100–1000)']


## Interactive Visualization

Use the **dropdown** to switch between datasets. The chart shows the proportion of each class per day as a stacked area chart.

In [5]:
COLOR_MAP = {
    "Negative":              "#EF553B",
    "Neutral":               "#636EFA",
    "Positive":              "#00CC96",
    "Low (0\u2013100)":      "#FFA15A",
    "High (100\u20131000)": "#AB63FA",
}

dataset_names = list(proportions.keys())
default_dataset = "global_covid19_tweets"

fig = go.Figure()

trace_indices: dict[str, list[int]] = {}
idx = 0

for ds_name in dataset_names:
    props = proportions[ds_name]
    trace_indices[ds_name] = []
    is_default = ds_name == default_dataset

    for col in props.columns:
        fig.add_trace(go.Scatter(
            x=props.index,
            y=props[col],
            name=col,
            mode="lines",
            stackgroup=ds_name,
            line=dict(width=0.5, color=COLOR_MAP.get(col, None)),
            hovertemplate=f"<b>{col}</b><br>Date: %{{x|%Y-%m-%d}}<br>Proportion: %{{y:.2%}}<extra></extra>",
            visible=is_default,
        ))
        trace_indices[ds_name].append(idx)
        idx += 1

buttons = []
for ds_name in dataset_names:
    visibility = [False] * idx
    for ti in trace_indices[ds_name]:
        visibility[ti] = True

    buttons.append(dict(
        label=ds_name,
        method="update",
        args=[
            {"visible": visibility},
            {"title": f"Class Proportions Over Time \u2014 {ds_name}"},
        ],
    ))

fig.update_layout(
    title=f"Class Proportions Over Time \u2014 {default_dataset}",
    xaxis_title="Date",
    yaxis_title="Proportion",
    yaxis=dict(tickformat=".0%", range=[0, 1]),
    hovermode="x unified",
    template="plotly_white",
    legend_title="Class",
    updatemenus=[
        dict(
            active=dataset_names.index(default_dataset),
            buttons=buttons,
            direction="down",
            showactive=True,
            x=0.0,
            xanchor="left",
            y=1.18,
            yanchor="top",
        )
    ],
    margin=dict(t=100),
)

fig.show()

## Confusion Matrix classifier behaviour across time chunks

In [8]:
cm_files = sorted(Path("../project/output_files").glob("confusion_matrices_*.csv"))

datasets = {}
for f in cm_files:
    label = f.stem.replace("confusion_matrices_", "")
    df = pd.read_csv(f)
    cm_cols = [c for c in df.columns if c.startswith("true_") and "_pred_" in c]
    df[cm_cols] = df[cm_cols].div(df[cm_cols].sum(axis=1), axis=0)
    datasets[label] = (df, cm_cols)

fig = go.Figure()
labels = list(datasets.keys())
trace_counts = []

for label, (df, cm_cols) in datasets.items():
    for col in cm_cols:
        fig.add_trace(go.Scatter(
            x=df["chunk_idx"],
            y=df[col],
            mode="lines+markers",
            name=col,
            visible=(label == labels[0]),
        ))
    trace_counts.append(len(cm_cols))

buttons = []
for i, label in enumerate(labels):
    visibility = []
    for j, count in enumerate(trace_counts):
        visibility.extend([i == j] * count)
    buttons.append(dict(label=label, method="update", args=[
        {"visible": visibility},
        {"title": f"Confusion Matrix Values Over Time — {label}"},
    ]))

fig.update_layout(
    title=f"Confusion Matrix Values Over Time — {labels[0]}",
    xaxis_title="Chunk Index",
    yaxis_title="Proportion",
    legend_title="CM Cell",
    hovermode="x unified",
    template="plotly_white",
    updatemenus=[dict(
        active=0,
        buttons=buttons,
        x=0.0,
        xanchor="left",
        y=1.15,
        yanchor="top",
    )],
)
fig.show()